# Submission: exp40 stacking without ConvNeXt-Tiny, C = 0.1, normalized weights, plateau thresholds

Card: [exp40_stacking_no_convnext.md](../../experiments/exp40_stacking_no_convnext.md)

In [ ]:
import os
import tempfile
from pathlib import Path

import pandas as pd

from core.config import ROOT, SUBMISSIONS
from core.cv import Config
from core.data import read_test
from core.submission import make_stacking_submission, submit

SMOKE = os.environ.get("SMOKE") == "1"
CONFIG_DIR = ROOT / "sections" / "model" / "configs"
CONFIGS = [
    Config.from_toml(CONFIG_DIR / "exp26_coarse_dropout_5fold" / "resnet18.toml"),
    Config.from_toml(CONFIG_DIR / "exp32_efficientnet_b0_5fold" / "efficientnet_b0.toml"),
    Config.from_toml(CONFIG_DIR / "exp33_regnety_016" / "regnety_016.toml"),
]
C = 0.1  # L2 strength, exp39 conclusion
NORMALIZE = True  # logit divided by sum of member weights per class
MARGIN = 0.05  # plateau: F1 >= max - MARGIN (exp37, exp38)
if SMOKE:
    TEST = read_test().head(16)
    PATH = Path(tempfile.mkdtemp()) / "smoke.csv"
else:
    TEST = None
    PATH = SUBMISSIONS / "ensemble-3-stacking-c01-normalized__plateau.csv"

## Predict

In [ ]:
path = make_stacking_submission(CONFIGS, PATH, C, test=TEST, margin=MARGIN, normalize=NORMALIZE)
pd.read_csv(path).iloc[:, 1:].mean().round(3)  # predicted positive rate per class

## Upload

In [ ]:
if not SMOKE:
    submit(path, "exp40 logistic regression stacking C=0.1, weights normalized per class, 3 models (exp26, EfficientNet-B0, RegNetY-016), 5-fold, OOF plateau thresholds")